# 4. C++ and CUDA

## What this notebook is about

You can write a method, or one step of one, in C++ or with your own CUDA kernel, and check
it exactly like Python code. There is no CMake or `setup.py`: one source file is enough.
When you pass a `.cpp` or `.cu` file, `svdlab` compiles it with PyTorch's own extension
builder (`torch.utils.cpp_extension`), imports the result as a Python module and calls its
`svd` function with PyTorch tensors. Your C++ code receives a `torch::Tensor`, so it can use
the whole ATen tensor library (the C++ side of PyTorch), raw pointers, or CUDA kernels.

The first build of a file takes 30 to 60 seconds. The result is cached and reused until the
file changes. Cells that need a tool you don't have print what to install and skip, so the
notebook always runs to the end.

## What you will learn

1. What has to be installed, and how to ask `svdlab` what is missing.
2. Step 1: check the toolchain with a minimal C++ method.
3. Step 2: a real algorithm ported to ATen, compared with its Python original.
4. Step 3: write and check your own `.cpp` file.
5. How to use a C++ function as a step inside a Python recipe.
6. Step 4: a hand-written CUDA kernel (GPU only).

## Words used here

| Word | Meaning |
| --- | --- |
| **native method** | a method in a `.cpp` / `.cc` (C++) or `.cu` (CUDA) file |
| **ATen** | PyTorch's C++ tensor library: the same operations as in Python (`matmul`, `index_select`, `linalg_svd`, ...), on CPU and GPU |
| **pybind11** | the library that makes a C++ function callable from Python; one line, `m.def("svd", &svd)`, does it |
| **ninja** | the small build tool PyTorch uses to compile the file |
| **nvcc** | NVIDIA's CUDA compiler, needed only for `.cu` files |

Text version: [tutorial 4](../docs/tutorials/04-native-cpp-cuda.md).

## Setup

**What this cell does:** it finds the repository folder (so the notebook works from any
working directory), makes `svdlab` importable even if you have not installed it with pip,
and chooses where to compute: `DEVICE` is `'cuda'` when PyTorch sees a GPU and `'cpu'`
otherwise. To stay on the CPU even with a GPU present, set `USE_GPU = False`.

**What you should see:** one line with the PyTorch version and the chosen device. If you
get `ModuleNotFoundError`, the notebook kernel is not the Python environment where you ran
`pip install -e '.[laboratory]'`. Choose that environment as the kernel and run the cell
again.

In [1]:
# Setup: find the repository, choose the device. Run this cell first.
from pathlib import Path
import os, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'svdlab' / '__init__.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))      # works even if the package is not installed with pip
os.chdir(ROOT)                         # all paths below are relative to the repository root

import torch
import svdlab

USE_GPU = True                         # set to False to stay on the CPU even when a GPU is present
DEVICE = 'cuda' if USE_GPU and torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)               # same CPU thread count as the command line, comparable timings
print(f'torch {torch.__version__}, device: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else ''))

torch 2.14.1+cu130, device: cpu


## What is needed, and what is missing here

| Tool | Install | Needed for |
| --- | --- | --- |
| ninja | `pip install ninja` | `.cpp`, `.cu` |
| Python headers (`Python.h`) | Debian/Ubuntu: `sudo apt install python3-dev` | `.cpp`, `.cu` |
| a C++17 compiler | `g++` or `clang++` | `.cpp`, `.cu` |
| CUDA toolkit with `nvcc`, matching `torch.version.cuda` | NVIDIA installer, or set `CUDA_HOME` | `.cu` only |

**What this cell does:** it asks `svdlab.native.missing_tools()` what is missing for a C++
build and for a CUDA build, and sets two flags, `HAVE_CPP` and `HAVE_CUDA`, that the later
cells use to decide whether to run or skip.

**What you should see:** `C++ build: ready` on most Linux and macOS machines with a
compiler. `CUDA build: ready` only on a machine with an NVIDIA GPU and the CUDA toolkit.

In [2]:
from svdlab import native

missing_cpp = native.missing_tools()
missing_cuda = native.missing_tools(cuda=True)
HAVE_CPP = not missing_cpp
HAVE_CUDA = HAVE_CPP and not missing_cuda and DEVICE == 'cuda'
print('C++ build :', 'ready' if HAVE_CPP else 'missing ' + '; '.join(missing_cpp))
print('CUDA build:', 'ready' if HAVE_CUDA else
      'not available (' + ('; '.join(missing_cuda) or 'no GPU visible to torch') + ')')

C++ build : ready
CUDA build: not available (nvcc; install the CUDA toolkit or set CUDA_HOME (e.g. /usr/local/cuda))


## Step 1. Check the toolchain

The C++ form of the contract:

```cpp
#include <torch/extension.h>

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor> svd(torch::Tensor a, int64_t rank) {
  ...                                   // same rules as in Python: U (m, k), s (k,), Vh (k, n)
  return {u, s, vh};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("svd", &svd); }   // makes svd callable from Python
```

`examples/native/minimal.cpp` simply calls the library SVD, so if it builds, every case
passes. That separates "my build setup is broken" from "my algorithm is wrong". The file
also exports a second function, `qr`, which we use in a later step.

**What this cell does:** it prints the file so you can read it.

In [3]:
print(Path('examples/native/minimal.cpp').read_text())

// The smallest native method: checks that your C++ toolchain works, and shows how to export
// a building block next to the SVD.
//
//     python -m svdlab check examples/native/minimal.cpp --quick
//
// svd() calls the library SVD (LAPACK on CPU tensors, cuSOLVER on GPU tensors), so every case
// should be ok. qr() has the contract of svdlab.blocks.qr and can be plugged into a recipe:
//
//     from svdlab import native, recipes
//     module = native.build('examples/native/minimal.cpp')
//     recipes.randomized(a, rank, qr=module.qr)
#include <torch/extension.h>

#include <tuple>

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor> svd(torch::Tensor a, int64_t rank) {
  auto [u, s, vh] = torch::linalg_svd(a, /*full_matrices=*/false);
  const int64_t k = std::min<int64_t>(rank, s.size(0));
  return {u.narrow(1, 0, k), s.narrow(0, 0, k), vh.narrow(0, 0, k)};
}

// Tall a (m, n) -> Q (m, n) with orthonormal columns, R (n, n) upper triangular.
std::tuple<torch::Tensor, torch::Tenso

**What this cell does:** `native.load(path)` compiles the file (only the first time) and
returns its `svd` as a Python function. Then it is checked like any method.

**What you should see:** after the build, 6/6 ok. If the build fails, the compiler error is
printed. The troubleshooting table at the end lists the usual causes.

In [4]:
if HAVE_CPP:
    minimal = native.load('examples/native/minimal.cpp')      # compiles on the first call
    rows = svdlab.check(minimal, svdlab.QUICK_SUITE, device=DEVICE)
else:
    print('Skipped: install', '; '.join(missing_cpp))

minimal              gaussian_tall          ok       0.378 ms  recon 1.55e-15  tail× -  σ-rel 1.78e-15  orth 1.23e-15
minimal              gaussian_wide          ok       0.120 ms  recon 1.66e-15  tail× -  σ-rel 1.44e-15  orth 1.29e-15


minimal              ill_conditioned_1e8    ok       0.176 ms  recon 4.40e-15  tail× -  σ-rel 2.10e-10  orth 1.49e-15
minimal              clustered              ok       0.117 ms  recon 2.65e-15  tail× -  σ-rel 2.26e-14  orth 1.36e-15


minimal              complex_gaussian       ok       0.157 ms  recon 2.34e-15  tail× -  σ-rel 1.33e-15  orth 1.23e-15
minimal              low_rank_k4            ok       0.280 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 5.55e-16  orth 5.09e-16


## Step 2. A real algorithm in ATen

`examples/native/my_svd.cpp` is the Jacobi SVD of `examples/my_svd.py` translated line by
line into ATen calls (`index_select`, `index_copy_`, `torch::where`, ...). Because it uses
only tensor operations, the same compiled code runs on CPU and GPU tensors, real and
complex. Translating a working Python prototype first, and checking that the numbers match,
is the recommended way into C++.

**What this cell does:** it compares the C++ file and the Python file on the quick suite
(file names are methods too) and shows the times side by side.

**What you should see:** both 6/6 ok with the same worst error, and the C++ version faster,
because the Python interpreter overhead per operation is gone.

In [5]:
if HAVE_CPP:
    rows = svdlab.compare({'my_svd.cpp': 'examples/native/my_svd.cpp', 'my_svd.py': 'examples/my_svd.py'},
                          svdlab.QUICK_SUITE, device=DEVICE, verbose=False)
    display(svdlab.table(rows).pivot(index='case', columns='method', values='time_median_ms').round(2))
    for s in svdlab.summary(rows):
        print(f"{s['method']:<11} {s['ok']}/{s['cases']} ok   worst recon {s['worst_reconstruction']:.2e}")
else:
    print('Skipped: install', '; '.join(missing_cpp))

method,my_svd.cpp,my_svd.py
case,,
clustered,37.45,69.81
complex_gaussian,19.78,32.91
gaussian_tall,29.05,73.44
gaussian_wide,17.81,39.54
ill_conditioned_1e8,44.64,85.77
low_rank_k4,107.82,172.49


my_svd.cpp  6/6 ok   worst recon 6.64e-06
my_svd.py   6/6 ok   worst recon 6.64e-06


## Step 3. Your turn: write a .cpp file

**What these cells do:** the first creates `laboratory_runs/` (ignored by git). The second
writes a small C++ SVD to `laboratory_runs/my_native_svd.cpp`: QR with the library, then the
library SVD of the small R, then U = Q Ur, with the wide-matrix trick from notebook 2. Edit
it and run these cells again. The build is redone automatically because the file changed.
Then the terminal helper is defined, and the file is checked from the command line, next to
`torch`.

**What you should see:** `Writing laboratory_runs/my_native_svd.cpp`, a build (about half a
minute), and 6/6 ok with `exit code: 0`.

In [6]:
Path('laboratory_runs').mkdir(exist_ok=True)

In [7]:
%%writefile laboratory_runs/my_native_svd.cpp
#include <torch/extension.h>

#include <tuple>

using torch::Tensor;

// a = Q R (library QR), R = Ur diag(s) Vh (library SVD of the small R), U = Q Ur.
std::tuple<Tensor, Tensor, Tensor> svd(Tensor a, int64_t rank) {
  TORCH_CHECK(a.dim() == 2, "expected one matrix");  // the message appears in the report
  const bool wide = a.size(0) < a.size(1);
  Tensor x = wide ? a.mH() : a;                      // work on a tall matrix
  auto [q, r] = torch::linalg_qr(x, "reduced");
  auto [ur, s, vh] = torch::linalg_svd(r, /*full_matrices=*/false);
  Tensor u = q.matmul(ur);
  const int64_t k = std::min<int64_t>(rank, s.size(0));
  u = u.narrow(1, 0, k);
  s = s.narrow(0, 0, k);
  vh = vh.narrow(0, 0, k);
  if (wide) return {vh.mH().contiguous(), s, u.mH().contiguous()};   // a = (xᴴ)ᴴ: swap the factors
  return {u, s, vh};
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("svd", &svd, "QR, then the SVD of R"); }

Writing laboratory_runs/my_native_svd.cpp


In [8]:
import shlex
from svdlab.__main__ import main as svdlab_main

def svdlab_cli(command):
    """Run `python -m svdlab <command>` inside this notebook (same output, same exit code)."""
    code = svdlab_main(shlex.split(command))
    print(f'exit code: {code}')

In [9]:
if HAVE_CPP:
    svdlab_cli(f'check laboratory_runs/my_native_svd.cpp --quick --device {DEVICE} --compare torch')
else:
    print('Skipped: install', '; '.join(missing_cpp))

recon  = |A - U diag(s) Vh| / |A|          tail× = recon / best possible (truncated cases)
σ-rel  = max relative error of the values   orth  = max(|UᴴU - I|, |V Vᴴ - I|) / sqrt(k)
ok     = orth and recon within 100·eps·max(m, n), or tail× <= 1.05



my_native_svd.cpp    gaussian_tall          ok       0.441 ms  recon 1.55e-15  tail× -  σ-rel 1.78e-15  orth 1.23e-15


my_native_svd.cpp    gaussian_wide          ok       0.190 ms  recon 2.59e-15  tail× -  σ-rel 2.78e-15  orth 1.53e-15


my_native_svd.cpp    ill_conditioned_1e8    ok       0.311 ms  recon 4.55e-15  tail× -  σ-rel 2.04e-10  orth 1.14e-15


my_native_svd.cpp    clustered              ok       0.399 ms  recon 1.68e-15  tail× -  σ-rel 6.07e-15  orth 1.21e-15


...

my_native_svd.cpp    low_rank_k4            ok       0.654 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 7.18e-16  orth 7.52e-16


torch                gaussian_tall          ok       0.447 ms  recon 1.55e-15  tail× -  σ-rel 1.78e-15  orth 1.23e-15


torch                gaussian_wide          ok       0.157 ms  recon 1.66e-15  tail× -  σ-rel 1.44e-15  orth 1.29e-15
torch                ill_conditioned_1e8    ok       0.169 ms  recon 4.40e-15  tail× -  σ-rel 2.10e-10  orth 1.49e-15


torch                clustered              ok       0.145 ms  recon 2.65e-15  tail× -  σ-rel 2.26e-14  orth 1.36e-15
torch                complex_gaussian       ok       0.178 ms  recon 2.34e-15  tail× -  σ-rel 1.33e-15  orth 1.23e-15


torch                low_rank_k4            ok       4.620 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 5.55e-16  orth 5.09e-16

my_native_svd.cpp: 6/6 ok, total median 2.200 ms
torch: 6/6 ok, total median 5.716 ms
exit code: 0


## A C++ block inside a Python recipe

`native.load` returns only the compiled `svd`. `native.build` returns the whole compiled
module, so any function the file exports can be used, for example as a step of a recipe
from notebook 3. `minimal.cpp` exports `qr` with the same contract as `blocks.qr`.

**What this cell does:** it gets the module (already compiled above, so this is instant)
and runs randomized SVD with the C++ `qr` inside.

**What you should see:** 6/6 ok. The QR now runs in C++, while the rest of the recipe is
still Python.

In [10]:
if HAVE_CPP:
    from svdlab import recipes
    module = native.build('examples/native/minimal.cpp')     # already compiled above: instant

    def randomized_cpp_qr(a, rank):
        return recipes.randomized(a, rank, qr=module.qr)

    rows = svdlab.check(randomized_cpp_qr, svdlab.QUICK_SUITE, device=DEVICE, repeat=3)
else:
    print('Skipped: install', '; '.join(missing_cpp))

randomized_cpp_qr    gaussian_tall          ok      61.614 ms  recon 5.43e-15  tail× -  σ-rel 4.56e-15  orth 5.57e-15


randomized_cpp_qr    gaussian_wide          ok      74.147 ms  recon 4.53e-15  tail× -  σ-rel 4.06e-15  orth 4.83e-15


randomized_cpp_qr    ill_conditioned_1e8    ok     131.687 ms  recon 7.18e-15  tail× -  σ-rel 2.18e-10  orth 8.74e-15


randomized_cpp_qr    clustered              ok     140.391 ms  recon 1.02e-14  tail× -  σ-rel 1.47e-14  orth 9.75e-15


randomized_cpp_qr    complex_gaussian       ok      35.880 ms  recon 3.86e-15  tail× -  σ-rel 3.47e-15  orth 3.82e-15
randomized_cpp_qr    low_rank_k4            ok      11.541 ms  recon 6.64e-06  tail× 1.00e+00  σ-rel 6.94e-16  orth 1.28e-15


## Step 4. Your own CUDA kernel (GPU only)

`examples/native/my_svd.cu` runs the Jacobi sweeps in a hand-written kernel. Each thread
block handles one pair of columns: it computes their three inner products with a
shared-memory reduction and applies the rotation. The host code loops over sweeps and
launches the kernel on PyTorch's current CUDA stream, so timings measure it correctly.
Things to copy from it for your own kernels:

- `TORCH_CHECK(condition, "message")` for input checks; the message reaches the report.
- `at::cuda::getCurrentCUDAStream()` for launches, `C10_CUDA_KERNEL_LAUNCH_CHECK()` after them.
- `AT_DISPATCH_FLOATING_TYPES` to support float32 and float64 with one template.
- Keep results on the GPU; returning CPU tensors fails the GPU check.

**What this cell does:** on a GPU machine with nvcc it compiles the kernel and compares it
with `torch`. Elsewhere it explains what is needed and skips.

**What you should see (GPU):** `ok` on the real cases, and `ERROR` on `complex_gaussian`.
That one is expected: the example kernel supports real numbers only, and it says so through
`TORCH_CHECK`.

In [11]:
if HAVE_CUDA:
    cuda_svd = native.load('examples/native/my_svd.cu')
    rows = svdlab.compare({'my_svd.cu': cuda_svd, 'torch': 'torch'}, svdlab.QUICK_SUITE, device=DEVICE)
else:
    print('Skipped: needs an NVIDIA GPU and the CUDA toolkit (nvcc).',
          'On a GPU machine: python -m svdlab check examples/native/my_svd.cu --device cuda --compare torch')

Skipped: needs an NVIDIA GPU and the CUDA toolkit (nvcc). On a GPU machine: python -m svdlab check examples/native/my_svd.cu --device cuda --compare torch


## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `Python headers` missing | `sudo apt install python3-dev` (matching your Python version) |
| `ninja` missing | `pip install ninja` in the kernel's environment |
| `nvcc` missing | install the CUDA toolkit matching `torch.version.cuda`, or set `CUDA_HOME` |
| a compiler error | it is printed in full; `native.build(path, verbose=True)` also shows the commands |
| a change is not picked up | builds are keyed on the file contents; included headers are not, so touch the `.cpp` |
| `does not export 'svd'` | add `m.def("svd", &svd);` inside `PYBIND11_MODULE(TORCH_EXTENSION_NAME, m)` |
| `output_device` is `cpu` on a GPU run | the code moved data to the host; create tensors with `a.options()` |

## Summary

- One `.cpp` or `.cu` file with `svd(a, rank)` and a `PYBIND11_MODULE` line is a method.
- Start with `minimal.cpp` to test the toolchain, port a Python prototype to ATen, then
  optimize with kernels.
- `native.build` gives you every exported function, for example a C++ step for a recipe.

## Next

[05_metrics_and_matrices.ipynb](05_metrics_and_matrices.ipynb): what every number means,
and your own test matrices.